In [ ]:
"""
LCO archive: paginated frame fetch + FITS header collection.

Usage in a notebook:
    from lco_fetch import fetch_all_frames, build_dataframe
    frames = fetch_all_frames('2026-08-01', '2026-08-08')
    df = build_dataframe(frames)
"""

'''
Please note that this code is generated using Claude AI, which 
permission to do so by LCO!
'''
import os
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [2]:

BASE = "https://archive-api.lco.global"

# Put the token in an environment variable instead of in the notebook:
#   Windows (PowerShell):  setx LCO_API_TOKEN "your_token"   (then restart Jupyter)
API_TOKEN = os.environ.get("LCO_API_TOKEN")
if not API_TOKEN:
    raise RuntimeError("Set the LCO_API_TOKEN environment variable first.")

# One shared session: reuses connections and retries on rate limits / server errors.
session = requests.Session()
session.headers.update({"Authorization": f"Token {API_TOKEN}"})
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)))

RuntimeError: Set the LCO_API_TOKEN environment variable first.

In [ ]:
# ---------------------------------------------------------------- frames ----
def fetch_frames_window(start, end, extra_params=None, limit=1000):
    """Yield every frame between start and end using cursor pagination."""
    params = {
        "start": start,
        "end": end,
        "reduction_level": 91,
        "configuration_type": "EXPOSE",   # filter on the server, not after download
        "limit": limit,
        "pagination_style": "cursor",
    }
    params.update(extra_params or {})

    url = f"{BASE}/frames/"
    while url:
        r = session.get(url, params=params if url.endswith("/frames/") else None, timeout=60)
        r.raise_for_status()
        page = r.json()
        yield from page["results"]
        url = page.get("next")   # full URL incl. cursor; None on the last page


In [ ]:
# def fetch_all_frames(start, end, window_days=1, extra_params=None):
#     """
#     Walk a date range in 1-day windows (more robust than one giant query),
#     de-duplicating by frame id in case windows overlap at the edges.
#     """
#     t0 = datetime.fromisoformat(start)
#     t1 = datetime.fromisoformat(end)
#     seen, frames = set(), []

#     while t0 < t1:
#         t_next = min(t0 + timedelta(days=window_days), t1)
#         w_start = t0.strftime("%Y-%m-%dT%H:%M:%SZ")
#         w_end = t_next.strftime("%Y-%m-%dT%H:%M:%SZ")
#         n_before = len(frames)
#         for f in fetch_frames_window(w_start, w_end, extra_params):
#             if f["id"] not in seen:
#                 seen.add(f["id"])
#                 frames.append(f)
#         print(f"{w_start} -> {w_end}: +{len(frames) - n_before} (total {len(frames)})")
#         t0 = t_next
#     return frames

In [ ]:
def fetch_all_frames(start, end, window_days=1, extra_params=None,
                     max_tries=8, max_wait=1800, giveup_limit_hours=6,
                     cache_dir="lco_frames_cache", passes=3, pass_pause=300):
    """
    Walk a date range in 1-day windows. Built to survive overnight runs:
      * timeouts / connection errors / server errors -> wait (exponential backoff) and retry
      * 401/403/429 (limit reached) -> wait 5, 10, 20, 30... min for it to reset, then keep going
      * a window that still fails is skipped, then retried in later passes
      * each finished window is saved to disk, so re-running skips completed days
    """
    os.makedirs(cache_dir, exist_ok=True)

    # Build the list of windows
    t0, t1 = datetime.fromisoformat(start), datetime.fromisoformat(end)
    windows = []
    while t0 < t1:
        t_next = min(t0 + timedelta(days=window_days), t1)
        windows.append((t0, t_next))
        t0 = t_next

    def get_window(w0, w1):
        """Fetch one window with retries. Returns list of frames, or None if it gave up."""
        w_start = w0.strftime("%Y-%m-%dT%H:%M:%SZ")
        w_end = w1.strftime("%Y-%m-%dT%H:%M:%SZ")
        path = os.path.join(cache_dir, f"{w0:%Y-%m-%dT%H%M%S}_{w1:%Y-%m-%dT%H%M%S}.json")

        if os.path.exists(path):                      # already done in an earlier run
            try:
                with open(path, encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError):
                os.remove(path)                       # corrupt file, re-fetch

        tries, limit_strikes, limit_started = 0, 0, None
        while True:
            try:
                # list() so a failure mid-pagination never leaves a half-finished window
                result = list(fetch_frames_window(w_start, w_end, extra_params))
                break
            except requests.exceptions.HTTPError as e:
                status = e.response.status_code if e.response is not None else None
                if status in (401, 403, 429):         # limit reached: wait for reset
                    limit_started = limit_started or time.time()
                    if time.time() - limit_started > giveup_limit_hours * 3600:
                        print(f"  Blocked for over {giveup_limit_hours}h, giving up on {w_start}")
                        return None
                    wait = min(max_wait, 300 * 2 ** limit_strikes)
                    limit_strikes += 1
                    print(f"  LIMIT HIT (HTTP {status}) at {datetime.now():%H:%M:%S}. "
                          f"Waiting {wait / 60:.0f} min, then retrying {w_start}")
                    time.sleep(wait)
                    continue
                reason = f"HTTP {status}"
            except (requests.exceptions.RequestException, ValueError) as e:
                reason = f"{type(e).__name__}"        # timeouts, dropped connections, bad JSON

            tries += 1
            if tries >= max_tries:
                print(f"  Giving up on {w_start} for now ({reason})")
                return None
            wait = min(max_wait, 5 * 2 ** tries)
            print(f"  {reason} on {w_start}. Retry {tries}/{max_tries - 1} in {wait}s")
            time.sleep(wait)

        # Cache only windows that are fully in the past (a still-running day would be incomplete)
        if w1 <= datetime.utcnow():
            with open(path + ".tmp", "w", encoding="utf-8") as f:
                json.dump(result, f)
            os.replace(path + ".tmp", path)
        return result

    # Several passes: windows that failed get another chance after a rest
    results, pending = {}, windows
    for p in range(1, passes + 1):
        still_failing = []
        for w0, w1 in pending:
            frames_w = get_window(w0, w1)
            if frames_w is None:
                still_failing.append((w0, w1))
            else:
                results[w0] = frames_w
                print(f"{w0:%Y-%m-%d}: {len(frames_w)} frames "
                      f"({len(results)}/{len(windows)} windows done)")
        pending = still_failing
        if not pending:
            break
        if p < passes:
            print(f"{len(pending)} window(s) failed in pass {p}. Resting {pass_pause}s, then retrying.")
            time.sleep(pass_pause)

    if pending:
        print("Still failed after all passes:", [f"{w0:%Y-%m-%d}" for w0, _ in pending])

    # Combine in date order, de-duplicating by frame id
    seen, frames = set(), []
    for w0 in sorted(results):
        for f in results[w0]:
            if f["id"] not in seen:
                seen.add(f["id"])
                frames.append(f)
    print(f"Total: {len(frames)} frames")
    return frames

In [ ]:
months = [
    ("2026-02-01T00:00:00", "2026-03-01T00:00:00"),
    ("2026-03-01T00:00:00", "2026-04-01T00:00:00"),
    ("2026-04-01T00:00:00", "2026-05-01T00:00:00"),
    ("2026-05-01T00:00:00", "2026-06-01T00:00:00"),
    ("2026-06-01T00:00:00", "2026-07-01T00:00:00"),
    ("2026-07-01T00:00:00", "2026-08-01T00:00:00"),
]

for start, end in months:
    frames = fetch_all_frames(start, end)
    df = build_dataframe(frames)
    print(f"Done {start[:7]}: {len(df)} total rows in cache\n")

In [ ]:
# --------------------------------------------------------------- headers ----
HEADER_FIELDS = ["WCSERR", "SATFRAC", "EXPTIME", "FILTER", "AIRMASS", "L1FWHM",
                 "MOONFRAC", "MOONDIST", "SKYMAG", "WMSHUMID", "WMSTEMP", "PIXSCALE"]
FRAME_FIELDS = ["basename", "observation_date", "site_id", "telescope_id", "instrument_id"]


In [ ]:
def _get_row(frame):
    r = session.get(f"{BASE}/frames/{frame['id']}/headers/", timeout=60)
    r.raise_for_status()
    h = r.json()["data"]
    row = {"id": frame["id"]}
    row.update({k: frame.get(k) for k in FRAME_FIELDS})
    row.update({k: h.get(k) for k in HEADER_FIELDS})   # missing keys become NaN, not dropped
    return row


In [ ]:
# def build_dataframe(frames, workers=8, cache_csv="lco_headers_cache.csv"):
#     """Fetch headers in parallel; resumes from a cache file if one exists."""
#     done = pd.read_csv(cache_csv) if os.path.exists(cache_csv) else pd.DataFrame()
#     done_ids = set(done["id"]) if len(done) else set()
#     todo = [f for f in frames if f["id"] not in done_ids]
#     print(f"{len(done_ids)} cached, {len(todo)} to fetch")

#     rows = []
#     with ThreadPoolExecutor(max_workers=workers) as ex:
#         for i, row in enumerate(ex.map(_get_row, todo), 1):
#             rows.append(row)
#             if i % 200 == 0:   # checkpoint so a crash doesn't lose everything
#                 pd.concat([done, pd.DataFrame(rows)]).to_csv(cache_csv, index=False)
#                 print(f"  {i}/{len(todo)}")

#     df = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
#     df = df.drop_duplicates(subset="id")
#     df.to_csv(cache_csv, index=False)
#     return df

In [ ]:
import os, time, random, threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests

# ---- settings (edit here) ----
MAX_TRIES = 8              # retries for timeouts / connection drops / 5xx
MAX_WAIT = 1800            # longest single wait (seconds)
LIMIT_WAIT_START = 300     # first pause when the limit is hit (5 min); then 10, 20, 30, 30...
LIMIT_GIVEUP = 6 * 3600    # stop if blocked this long with no success in between
LIMIT_GIVEUP_FIRST = 3600  # stop sooner if NOTHING has worked yet this run (likely a bad token)

COLUMNS = ["id"] + FRAME_FIELDS + HEADER_FIELDS

# ---- shared pause state: when the limit is hit, ALL worker threads wait together ----
_lock = threading.Lock()
_st = {"pause_until": 0.0, "strikes": 0, "limit_since": None, "had_success": False}

class FatalLimit(Exception):
    """Blocked for too long; stop cleanly (progress is saved)."""

class GaveUp(Exception):
    """One request failed after all retries (temporary problem); will be retried next pass."""


def _wait_if_paused():
    while _st["pause_until"] > time.time():
        time.sleep(min(_st["pause_until"] - time.time(), 30))


def _limit_pause(status):
    with _lock:
        now = time.time()
        if now < _st["pause_until"]:            # another thread already started this pause
            return
        _st["limit_since"] = _st["limit_since"] or now
        giveup = LIMIT_GIVEUP if _st["had_success"] else LIMIT_GIVEUP_FIRST
        if now - _st["limit_since"] > giveup:
            raise FatalLimit(f"Blocked (HTTP {status}) for {(now - _st['limit_since']) / 60:.0f} min "
                             "with no success. " + ("" if _st["had_success"] else "Check your API token."))
        wait = min(MAX_WAIT, LIMIT_WAIT_START * 2 ** _st["strikes"])
        _st["strikes"] += 1
        _st["pause_until"] = now + wait
        print(f"  LIMIT HIT (HTTP {status}). All workers pausing {wait / 60:.1f} min "
              f"(resume ~{datetime.fromtimestamp(now + wait):%H:%M:%S})")


def _get_json(url):
    """GET with retries. Limit errors (401/403/429) pause everyone and don't use up retries."""
    tries = 0
    while True:
        _wait_if_paused()
        try:
            r = session.get(url, timeout=(15, 90))
            if r.status_code in (401, 403, 429):
                _limit_pause(r.status_code)
                continue
            if r.status_code in (408, 500, 502, 503, 504):
                reason = f"HTTP {r.status_code}"
            elif r.status_code >= 400:
                raise GaveUp(f"HTTP {r.status_code}")      # e.g. 404: won't fix itself; skip for now
            else:
                data = r.json()
                with _lock:
                    _st.update(strikes=0, limit_since=None, had_success=True)
                return data
        except (FatalLimit, GaveUp):
            raise
        except (requests.exceptions.RequestException, ValueError) as e:
            reason = type(e).__name__                      # timeout, dropped connection, bad JSON
        tries += 1
        if tries >= MAX_TRIES:
            raise GaveUp(reason)
        time.sleep(min(MAX_WAIT, 5 * 2 ** tries) + random.uniform(0, 3))


def _get_row(frame):
    h = _get_json(f"{BASE}/frames/{frame['id']}/headers/")["data"]
    row = {"id": frame["id"]}
    row.update({k: frame.get(k) for k in FRAME_FIELDS})
    row.update({k: h.get(k) for k in HEADER_FIELDS})       # missing keys become NaN
    return row


def build_dataframe(frames, workers=8, cache_csv="lco_headers_cache.csv",
                    chunk=200, passes=3, pass_pause=300, failed_csv="lco_failed_ids.csv"):
    """Fetch headers in parallel. Overnight-safe: saves every chunk, survives errors, resumes on re-run."""
    _st.update(pause_until=0.0, strikes=0, limit_since=None, had_success=False)

    def load():
        if not os.path.exists(cache_csv):
            return pd.DataFrame(columns=COLUMNS)
        return pd.read_csv(cache_csv, on_bad_lines="skip").drop_duplicates(subset="id")

    done_ids = set(load()["id"])
    print(f"{len(done_ids)} cached")
    failed_log = []

    try:
        for p in range(1, passes + 1):
            todo = [f for f in frames if f["id"] not in done_ids]
            if not todo:
                break
            print(f"=== Pass {p}/{passes}: {len(todo)} to fetch ===")
            n_fail = 0
            for i in range(0, len(todo), chunk):
                batch, rows = todo[i:i + chunk], []
                ex = ThreadPoolExecutor(max_workers=workers)
                try:
                    futs = {ex.submit(_get_row, f): f for f in batch}
                    for fut in as_completed(futs):
                        try:
                            rows.append(fut.result())
                        except FatalLimit:
                            raise
                        except Exception as e:             # one bad frame never stops the run
                            n_fail += 1
                            failed_log.append((futs[fut]["id"], str(e)[:150],
                                               datetime.now().isoformat(timespec="seconds")))
                except BaseException:
                    ex.shutdown(wait=False, cancel_futures=True)
                    raise
                finally:                                   # always save what we got
                    if rows:
                        pd.DataFrame(rows).reindex(columns=COLUMNS).to_csv(
                            cache_csv, mode="a", header=not os.path.exists(cache_csv), index=False)
                        done_ids.update(r["id"] for r in rows)
                ex.shutdown(wait=True)
                print(f"  {min(i + chunk, len(todo))}/{len(todo)}  ({n_fail} failed so far this pass)")
            if n_fail and p < passes:
                print(f"Resting {pass_pause}s, then retrying the {n_fail} failures")
                time.sleep(pass_pause)
    except FatalLimit as e:
        print(f"STOPPING: {e}\nProgress is saved; re-run to resume.")
    except KeyboardInterrupt:
        print("Interrupted. Progress is saved; re-run to resume.")
    finally:
        left = [f for f in frames if f["id"] not in done_ids]
        if left:
            pd.DataFrame(failed_log, columns=["id", "reason", "time"]).to_csv(failed_csv, index=False)
            print(f"{len(left)} frames still missing (reasons in {failed_csv})")

    df = load()
    print(f"Done: {len(df)} rows in {cache_csv}")
    return df

In [ ]:
if __name__ == "__main__":
    frames = fetch_all_frames("2026-08-01T00:00:00", "2026-08-08T00:00:00")
    df = build_dataframe(frames)
    print(df.shape)
    print(df["WCSERR"].value_counts(dropna=False))